In [2]:
from pathlib import Path
import pandas as pd
import json

In [15]:
QUESTION_DIR = Path(r"C:\Users\user\Desktop\질문")

excel_files = [
    QUESTION_DIR / "김경민_평가질문작성.xlsx",
    QUESTION_DIR / "김지수_평가질문작성.xlsx",
    QUESTION_DIR / "남종희_평가질문작성.xlsx",
    QUESTION_DIR / "장세현_평가질문작성.xlsx",
    QUESTION_DIR / "조지민_평가질문작성.xlsx",
    QUESTION_DIR / "이승주_평가질문작성.xlsx",
    QUESTION_DIR / "윤찬울_평가질문작성.xlsx",
]

for file in excel_files:
    print(file.name, "존재:", file.exists())

김경민_평가질문작성.xlsx 존재: True
김지수_평가질문작성.xlsx 존재: True
남종희_평가질문작성.xlsx 존재: True
장세현_평가질문작성.xlsx 존재: True
조지민_평가질문작성.xlsx 존재: True
이승주_평가질문작성.xlsx 존재: True
윤찬울_평가질문작성.xlsx 존재: True


In [16]:
frames = []

for file in excel_files:
    df = pd.read_excel(
        file,
        sheet_name="질문작성",
        dtype={
            "id": "string",
            "region": "string",
            "expected_policy_id": "string",
            "answer": "string",
            "evidence": "string",
            "source": "string",
            "question_type": "string",
            "author": "string",
        },
    )

    df = df.dropna(subset=["question"]).copy()
    df["input_file"] = file.name
    frames.append(df)

eval_df = pd.concat(frames, ignore_index=True)

print("전체 문항 수:", len(eval_df))
display(eval_df.head())

전체 문항 수: 280


,id,region,age,question,expected_policy_id,answer,evidence,source,question_type,author,input_file
0,Q081,대전광역시,NaN,대전에서 1인 영세 자영업자 고용보험료 지원을 받으려면 매출액과 가입 조건이 어떻게...,local:WLF00000877,1인 영세 자영업자 고용보험료 지원,"[""사업자등록증상 사업장 소재지가 대전이며 연매출 3억원 이하인 1인 영세 자영업자...",data/raw/local/detail/WLF00000877.xml,직접형,팀원3,김경민_평가질문작성.xlsx
1,Q082,대전광역시,NaN,대전에 사업장이 있고 연매출이 2억 원인 고용보험 가입 1인 자영업자입니다. 보험료...,local:WLF00000877,1인 영세 자영업자 고용보험료 지원,"[""사업자등록증상 사업장 소재지가 대전이며 연매출 3억원 이하인 1인 영세 자영업자...",data/raw/local/detail/WLF00000877.xml,간접형,팀원3,김경민_평가질문작성.xlsx
2,Q083,대전광역시,NaN,대전 구직청년 면접용 정장대여의 나이와 취업 상태 요건은 무엇이고 어디에서 신청하나요?,local:WLF00004054,구직청년 면접용 정장대여,"[""대전시에 거주하는 18~39세 미취업 청년구직자"", ""미취업자 구직청년(마이스터...",data/raw/local/detail/WLF00004054.xml,직접형,팀원3,김경민_평가질문작성.xlsx
3,Q084,대전광역시,25.0,대전에 사는 25세 미취업 구직자입니다. 면접 정장이 필요한데 대전청년포털에서 인터...,local:WLF00004054,구직청년 면접용 정장대여,"[""대전시에 거주하는 18~39세 미취업 청년구직자"", ""미취업자 구직청년(마이스터...",data/raw/local/detail/WLF00004054.xml,간접형,팀원3,김경민_평가질문작성.xlsx
4,Q085,대전광역시,NaN,대전 청년 월세지원의 연령·무주택·보증금·월세·소득 요건과 온라인 신청처를 알려주세요.,local:WLF00005253,대전 청년 월세지원,"[""주민등록상 대전광역시에 거주하는 19~39세 이하의 무주택 청년 가구"", ""- ...",data/raw/local/detail/WLF00005253.xml,직접형,팀원3,김경민_평가질문작성.xlsx


In [17]:
required_columns = [
    "id",
    "region",
    "question",
    "expected_policy_id",
    "answer",
    "evidence",
    "source",
    "question_type",
    "author",
]

print("전체 열:", eval_df.columns.tolist())
print("전체 문항:", len(eval_df))
print("고유 정책 수:", eval_df["expected_policy_id"].nunique())

print("\n필수값 누락:")
print(eval_df[required_columns].isna().sum())

print("\n중복 ID 수:", eval_df["id"].duplicated().sum())
print("\n질문 유형:")
print(eval_df["question_type"].value_counts())


def validate_evidence(value):
    try:
        result = json.loads(value)

        if not isinstance(result, list):
            return "리스트가 아님"

        if len(result) == 0:
            return "빈 리스트"

        if not all(isinstance(item, str) for item in result):
            return "문자열이 아닌 항목 존재"

        return None

    except Exception as error:
        return str(error)


eval_df["evidence_error"] = eval_df["evidence"].apply(validate_evidence)

evidence_errors = eval_df[eval_df["evidence_error"].notna()][
    ["id", "evidence", "evidence_error", "input_file"]
]

print("\nevidence 오류 수:", len(evidence_errors))
display(evidence_errors)

전체 열: ['id', 'region', 'age', 'question', 'expected_policy_id', 'answer', 'evidence', 'source', 'question_type', 'author', 'input_file']
전체 문항: 280
고유 정책 수: 140

필수값 누락:
id                    0
region                0
question              0
expected_policy_id    0
answer                0
evidence              0
source                0
question_type         0
author                0
dtype: int64

중복 ID 수: 0

질문 유형:
question_type
직접형    140
간접형    140
Name: count, dtype: int64[pyarrow]

evidence 오류 수: 0


,id,evidence,evidence_error,input_file


In [18]:
import chromadb

PROJECT_DIR = Path(r"C:\ai_project01\WelfareOne")
OUTPUT_DIR = PROJECT_DIR / "data" / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "welfareone_rag_eval_280.csv"

# 검사용 임시 열 제거
save_df = eval_df.drop(
    columns=["evidence_error"],
    errors="ignore",
).copy()

# 25.0처럼 저장되지 않도록 정리
save_df["age"] = save_df["age"].astype("Int64")

save_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("저장 위치:", OUTPUT_CSV)
print("저장 문항 수:", len(save_df))

저장 위치: C:\ai_project01\WelfareOne\data\evaluation\welfareone_rag_eval_280.csv
저장 문항 수: 280


In [19]:
DB_PATH = PROJECT_DIR / "data" / "chroma"
COLLECTION_NAME = "welfare_policy_sections_qwen_v3"

client = chromadb.PersistentClient(path=str(DB_PATH))
collection = client.get_collection(COLLECTION_NAME)

policy_records = collection.get(
    where={"section": "candidate_summary"},
    include=["metadatas"],
)

db_policy_ids = {
    metadata["policy_id"]
    for metadata in policy_records["metadatas"]
    if metadata.get("policy_id")
}

expected_policy_ids = set(save_df["expected_policy_id"])

missing_policy_ids = sorted(expected_policy_ids - db_policy_ids)

print("ChromaDB 전체 레코드:", collection.count())
print("ChromaDB 정책 수:", len(db_policy_ids))
print("평가 대상 정책 수:", len(expected_policy_ids))
print("DB에 없는 정답 정책 수:", len(missing_policy_ids))

if missing_policy_ids:
    display(
        save_df[
            save_df["expected_policy_id"].isin(missing_policy_ids)
        ][
            [
                "id",
                "expected_policy_id",
                "answer",
                "input_file",
            ]
        ]
    )
else:
    print("모든 정답 정책이 ChromaDB에 존재합니다.")

ChromaDB 전체 레코드: 11850
ChromaDB 정책 수: 1418
평가 대상 정책 수: 140
DB에 없는 정답 정책 수: 0
모든 정답 정책이 ChromaDB에 존재합니다.


In [20]:
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np

DB_PATH = PROJECT_DIR / "data" / "chroma"
COLLECTION_NAME = "welfare_policy_sections_qwen_v3"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"

client = chromadb.PersistentClient(path=str(DB_PATH))
collection = client.get_collection(COLLECTION_NAME)

embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    local_files_only=True,
)

print("컬렉션:", COLLECTION_NAME)
print("전체 레코드:", collection.count())
print("임베딩 모델:", EMBEDDING_MODEL)
print("사용 장치:", embedder.device)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 645.49it/s]


컬렉션: welfare_policy_sections_qwen_v3
전체 레코드: 11850
임베딩 모델: Qwen/Qwen3-Embedding-0.6B
사용 장치: cuda:0


In [21]:
questions = save_df["question"].astype(str).tolist()

question_embeddings = embedder.encode(
    questions,
    prompt_name="query",
    normalize_embeddings=True,
    batch_size=16,
    show_progress_bar=True,
)

print("질문 수:", len(question_embeddings))
print("벡터 차원:", question_embeddings.shape[1])

Batches: 100%|██████████| 18/18 [00:00<00:00, 19.04it/s]

질문 수: 280
벡터 차원: 1024


In [22]:
VECTOR_TOP_K_CHUNKS = 100
FINAL_TOP_K_POLICIES = 5
QUERY_BATCH_SIZE = 20

vector_result_rows = []

for start in range(0, len(save_df), QUERY_BATCH_SIZE):
    end = min(start + QUERY_BATCH_SIZE, len(save_df))

    batch_embeddings = question_embeddings[start:end].tolist()

    batch_results = collection.query(
        query_embeddings=batch_embeddings,
        n_results=VECTOR_TOP_K_CHUNKS,
        where={"searchable": True},
        include=["metadatas", "distances"],
    )

    for offset in range(end - start):
        row = save_df.iloc[start + offset]

        metadatas = batch_results["metadatas"][offset]
        distances = batch_results["distances"][offset]

        # 같은 정책에서 여러 청크가 검색되면 가장 가까운 청크만 사용
        policy_candidates = {}

        for metadata, distance in zip(metadatas, distances):
            policy_id = metadata.get("policy_id")

            if not policy_id:
                continue

            previous = policy_candidates.get(policy_id)

            if previous is None or distance < previous["distance"]:
                policy_candidates[policy_id] = {
                    "policy_id": policy_id,
                    "title": metadata.get("title", ""),
                    "distance": float(distance),
                }

        # 순수 벡터 검색이므로 거리가 가까운 순서만 사용
        ranked_policies = sorted(
            policy_candidates.values(),
            key=lambda item: item["distance"],
        )

        top5 = ranked_policies[:FINAL_TOP_K_POLICIES]
        top5_ids = [item["policy_id"] for item in top5]
        top5_titles = [item["title"] for item in top5]
        top5_distances = [item["distance"] for item in top5]

        expected_id = row["expected_policy_id"]

        expected_rank = next(
            (
                rank
                for rank, item in enumerate(ranked_policies, start=1)
                if item["policy_id"] == expected_id
            ),
            None,
        )

        result_row = {
            "id": row["id"],
            "question": row["question"],
            "region": row["region"],
            "age": row["age"],
            "question_type": row["question_type"],
            "expected_policy_id": expected_id,
            "expected_policy_name": row["answer"],
            "expected_rank": expected_rank,
            "hit_at_1": int(expected_rank == 1),
            "hit_at_3": int(
                expected_rank is not None and expected_rank <= 3
            ),
            "hit_at_5": int(
                expected_rank is not None and expected_rank <= 5
            ),
            "reciprocal_rank": (
                1 / expected_rank if expected_rank is not None else 0
            ),
        }

        for rank in range(1, FINAL_TOP_K_POLICIES + 1):
            if rank <= len(top5):
                result_row[f"top{rank}_policy_id"] = top5_ids[rank - 1]
                result_row[f"top{rank}_title"] = top5_titles[rank - 1]
                result_row[f"top{rank}_distance"] = top5_distances[rank - 1]
            else:
                result_row[f"top{rank}_policy_id"] = None
                result_row[f"top{rank}_title"] = None
                result_row[f"top{rank}_distance"] = None

        vector_result_rows.append(result_row)

    print(f"검색 완료: {end}/{len(save_df)}")

vector_results_df = pd.DataFrame(vector_result_rows)

print("검색 완료 문항:", len(vector_results_df))
display(vector_results_df.head())

검색 완료: 20/280
검색 완료: 40/280
검색 완료: 60/280
검색 완료: 80/280
검색 완료: 100/280
검색 완료: 120/280
검색 완료: 140/280
검색 완료: 160/280
검색 완료: 180/280
검색 완료: 200/280
검색 완료: 220/280
검색 완료: 240/280
검색 완료: 260/280
검색 완료: 280/280
검색 완료 문항: 280


,id,question,region,age,question_type,expected_policy_id,expected_policy_name,expected_rank,hit_at_1,hit_at_3,...,top2_distance,top3_policy_id,top3_title,top3_distance,top4_policy_id,top4_title,top4_distance,top5_policy_id,top5_title,top5_distance
0,Q081,대전에서 1인 영세 자영업자 고용보험료 지원을 받으려면 매출액과 가입 조건이 어떻게...,대전광역시,<NA>,직접형,local:WLF00000877,1인 영세 자영업자 고용보험료 지원,1.0,1,1,...,0.357229,local:WLF00006567,1인 자영업자 등 출산급여 지원,0.417002,local:WLF00002542,국민건강보험료 및 장기요양보험료 지원,0.434673,local:WLF00005253,대전 청년 월세지원,0.442353
1,Q082,대전에 사업장이 있고 연매출이 2억 원인 고용보험 가입 1인 자영업자입니다. 보험료...,대전광역시,<NA>,간접형,local:WLF00000877,1인 영세 자영업자 고용보험료 지원,1.0,1,1,...,0.335401,local:WLF00006325,강북구 배달플랫폼 종사자 고용보험료 지원 사업,0.341407,national:WLF00006263,저소득 지역가입자 보험료 지원,0.357797,local:WLF00006518,플랫폼 배달노동자 산재보험 지원,0.362345
2,Q083,대전 구직청년 면접용 정장대여의 나이와 취업 상태 요건은 무엇이고 어디에서 신청하나요?,대전광역시,<NA>,직접형,local:WLF00004054,구직청년 면접용 정장대여,1.0,1,1,...,0.195492,local:WLF00004287,청년 면접정장 무료대여 지원사업,0.228836,local:WLF00005376,청년 면접정장 무료대여사업,0.232097,local:WLF00001493,용인청년 희망옷장(면접정장 무료대여) 사업,0.237904
3,Q084,대전에 사는 25세 미취업 구직자입니다. 면접 정장이 필요한데 대전청년포털에서 인터...,대전광역시,25,간접형,local:WLF00004054,구직청년 면접용 정장대여,1.0,1,1,...,0.278339,local:WLF00004287,청년 면접정장 무료대여 지원사업,0.298098,local:WLF00001622,평택청년면접정장무료대여서비스,0.302170,local:WLF00006136,취업준비청년 정장대여 사업,0.304845
4,Q085,대전 청년 월세지원의 연령·무주택·보증금·월세·소득 요건과 온라인 신청처를 알려주세요.,대전광역시,<NA>,직접형,local:WLF00005253,대전 청년 월세지원,1.0,1,1,...,0.234671,local:WLF00005696,청년 전월세보증금 대출이자 지원사업,0.243080,local:WLF00004712,평택시 청년 전월세보증금 대출이자 지원,0.257654,local:WLF00006135,청년 월세 지원,0.260302


In [23]:
def show_metrics(df, label):
    print(f"\n[{label}]")
    print("문항 수:", len(df))
    print("Hit@1:", round(df["hit_at_1"].mean(), 4))
    print("Hit@3:", round(df["hit_at_3"].mean(), 4))
    print("Hit@5:", round(df["hit_at_5"].mean(), 4))
    print("MRR:", round(df["reciprocal_rank"].mean(), 4))


show_metrics(vector_results_df, "전체")

for question_type in ["직접형", "간접형"]:
    subset = vector_results_df[
        vector_results_df["question_type"] == question_type
    ]
    show_metrics(subset, question_type)


VECTOR_RESULT_CSV = (
    OUTPUT_DIR / "rag_vector_only_results2.csv"
)

vector_results_df.to_csv(
    VECTOR_RESULT_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("\n결과 저장:", VECTOR_RESULT_CSV)


[전체]
문항 수: 280
Hit@1: 0.7857
Hit@3: 0.9
Hit@5: 0.9357
MRR: 0.8492

[직접형]
문항 수: 140
Hit@1: 0.9143
Hit@3: 0.9714
Hit@5: 0.9857
MRR: 0.9441

[간접형]
문항 수: 140
Hit@1: 0.6571
Hit@3: 0.8286
Hit@5: 0.8857
MRR: 0.7542

결과 저장: C:\ai_project01\WelfareOne\data\evaluation\rag_vector_only_results2.csv


In [24]:
import re


def normalize_region_text(value):
    """지역 비교를 위해 공백을 제거한다."""
    return re.sub(r"\s+", "", str(value or "")).strip()


# 현재 ChromaDB에 실제로 저장된 시·도 이름을 가져온다.
REGION_LEVEL1_VALUES = sorted(
    {
        str(metadata.get("region_level1", "")).strip()
        for metadata in policy_records["metadatas"]
        if str(metadata.get("region_level1", "")).strip()
    },
    key=len,
    reverse=True,
)

print("DB 시·도 목록:")
print(REGION_LEVEL1_VALUES)


def parse_eval_region(region_text):
    """
    평가 데이터의 region을 시·도와 시·군·구로 나눈다.

    예:
    부산광역시 해운대구
    → 부산광역시, 해운대구

    부산광역시
    → 부산광역시, None

    중앙부처 또는 전국
    → None, None
    """
    text = str(region_text or "").strip()

    if not text or text in {"전국", "중앙부처"}:
        return None, None

    normalized = normalize_region_text(text)

    region_level1 = next(
        (
            region
            for region in REGION_LEVEL1_VALUES
            if normalized.startswith(
                normalize_region_text(region)
            )
        ),
        None,
    )

    if region_level1 is None:
        raise ValueError(
            f"DB 공식 지역명과 일치하지 않는 시·도입니다: {text}"
        )

    remainder = normalized[
        len(normalize_region_text(region_level1)):
    ]

    region_level2 = remainder if remainder else None

    return region_level1, region_level2


def make_region_search_scopes(region_text):
    """
    사용자가 선택한 지역에 따라 ChromaDB 검색 조건을 만든다.

    시·군·구가 있는 경우:
    1. 해당 시·군·구
    2. 해당 시·도의 광역 정책
    3. 중앙부처

    시·도만 있는 경우:
    1. 해당 시·도 소속 정책
    2. 중앙부처
    """
    region_level1, region_level2 = parse_eval_region(
        region_text
    )

    scopes = []

    if region_level1 and region_level2:
        # 1. 정확한 시·군·구 정책
        scopes.append(
            {
                "scope_name": "district",
                "scope_label": (
                    f"{region_level1} {region_level2}"
                ),
                "where": {
                    "$and": [
                        {
                            "searchable": {
                                "$eq": True
                            }
                        },
                        {
                            "source": {
                                "$eq": "local"
                            }
                        },
                        {
                            "region_level1": {
                                "$eq": region_level1
                            }
                        },
                        {
                            "region_level2": {
                                "$eq": region_level2
                            }
                        },
                    ]
                },
            }
        )

        # 2. 시·군·구가 비어 있는 시·도 광역 정책
        scopes.append(
            {
                "scope_name": "province",
                "scope_label": (
                    f"{region_level1} 광역"
                ),
                "where": {
                    "$and": [
                        {
                            "searchable": {
                                "$eq": True
                            }
                        },
                        {
                            "source": {
                                "$eq": "local"
                            }
                        },
                        {
                            "region_level1": {
                                "$eq": region_level1
                            }
                        },
                        {
                            "region_level2": {
                                "$eq": ""
                            }
                        },
                    ]
                },
            }
        )

    elif region_level1:
        # 시·도만 입력된 경우 해당 시·도 전체 검색
        scopes.append(
            {
                "scope_name": "province",
                "scope_label": region_level1,
                "where": {
                    "$and": [
                        {
                            "searchable": {
                                "$eq": True
                            }
                        },
                        {
                            "source": {
                                "$eq": "local"
                            }
                        },
                        {
                            "region_level1": {
                                "$eq": region_level1
                            }
                        },
                    ]
                },
            }
        )

    # 3. 중앙부처 정책은 지역과 관계없이 검색
    scopes.append(
        {
            "scope_name": "national",
            "scope_label": "중앙부처",
            "where": {
                "$and": [
                    {
                        "searchable": {
                            "$eq": True
                        }
                    },
                    {
                        "source": {
                            "$eq": "national"
                        }
                    },
                ]
            },
        }
    )

    return region_level1, region_level2, scopes


# 지역 파싱 시험
for sample_region in [
    "부산광역시 해운대구",
    "서울특별시 관악구",
    "부산광역시",
    "중앙부처",
]:
    try:
        level1, level2, scopes = (
            make_region_search_scopes(sample_region)
        )

        print("\n입력:", sample_region)
        print("시·도:", level1)
        print("시·군·구:", level2)
        print(
            "검색 범위:",
            [
                scope["scope_label"]
                for scope in scopes
            ],
        )

    except Exception as error:
        print(
            "\n지역 파싱 실패:",
            sample_region,
            error,
        )

DB 시·도 목록:
['전남광주통합특별시', '세종특별자치시', '전북특별자치도', '제주특별자치도', '강원특별자치도', '대구광역시', '울산광역시', '인천광역시', '부산광역시', '서울특별시', '대전광역시', '경상북도', '경상남도', '충청북도', '충청남도', '경기도']

입력: 부산광역시 해운대구
시·도: 부산광역시
시·군·구: 해운대구
검색 범위: ['부산광역시 해운대구', '부산광역시 광역', '중앙부처']

입력: 서울특별시 관악구
시·도: 서울특별시
시·군·구: 관악구
검색 범위: ['서울특별시 관악구', '서울특별시 광역', '중앙부처']

입력: 부산광역시
시·도: 부산광역시
시·군·구: None
검색 범위: ['부산광역시', '중앙부처']

입력: 중앙부처
시·도: None
시·군·구: None
검색 범위: ['중앙부처']


In [25]:
REGION_VECTOR_TOP_K_CHUNKS = 100
REGION_FINAL_TOP_K_POLICIES = 5

region_result_rows = []


def add_scope_results(
    policy_candidates,
    embedding,
    scope,
):
    """
    하나의 지역 범위에서 벡터 검색하고
    정책별로 가장 가까운 청크만 저장한다.
    """
    result = collection.query(
        query_embeddings=[embedding.tolist()],
        n_results=REGION_VECTOR_TOP_K_CHUNKS,
        where=scope["where"],
        include=["metadatas", "distances"],
    )

    metadatas = result["metadatas"][0]
    distances = result["distances"][0]

    for metadata, distance in zip(
        metadatas,
        distances,
    ):
        policy_id = metadata.get("policy_id")

        if not policy_id:
            continue

        candidate = {
            "policy_id": policy_id,
            "title": metadata.get("title", ""),
            "distance": float(distance),
            "region_scope": scope["scope_name"],
            "region_scope_label": (
                scope["scope_label"]
            ),
        }

        previous = policy_candidates.get(
            policy_id
        )

        # 같은 정책에서 여러 청크가 검색되면
        # 거리가 가장 가까운 청크만 사용
        if (
            previous is None
            or candidate["distance"]
            < previous["distance"]
        ):
            policy_candidates[policy_id] = (
                candidate
            )


for position, (_, row) in enumerate(
    save_df.iterrows()
):
    question_embedding = (
        question_embeddings[position]
    )

    try:
        (
            region_level1,
            region_level2,
            search_scopes,
        ) = make_region_search_scopes(
            row["region"]
        )

        policy_candidates = {}

        # 시군구, 광역, 중앙부처를 각각 검색
        for scope in search_scopes:
            add_scope_results(
                policy_candidates,
                question_embedding,
                scope,
            )

        # 지역 필터 안에 포함된 정책들을
        # 순수 벡터 거리순으로 정렬
        ranked_policies = sorted(
            policy_candidates.values(),
            key=lambda item: item["distance"],
        )

        expected_id = row[
            "expected_policy_id"
        ]

        expected_rank = next(
            (
                rank
                for rank, item in enumerate(
                    ranked_policies,
                    start=1,
                )
                if item["policy_id"]
                == expected_id
            ),
            None,
        )

        top5 = ranked_policies[
            :REGION_FINAL_TOP_K_POLICIES
        ]

        result_row = {
            "id": row["id"],
            "question": row["question"],
            "region": row["region"],
            "input_region_level1": (
                region_level1
            ),
            "input_region_level2": (
                region_level2
            ),
            "age": row["age"],
            "question_type": (
                row["question_type"]
            ),
            "expected_policy_id": (
                expected_id
            ),
            "expected_policy_name": (
                row["answer"]
            ),
            "expected_rank": (
                expected_rank
            ),
            "hit_at_1": int(
                expected_rank == 1
            ),
            "hit_at_3": int(
                expected_rank is not None
                and expected_rank <= 3
            ),
            "hit_at_5": int(
                expected_rank is not None
                and expected_rank <= 5
            ),
            "reciprocal_rank": (
                1 / expected_rank
                if expected_rank is not None
                else 0.0
            ),
            "region_error": None,
        }

        for rank in range(
            1,
            REGION_FINAL_TOP_K_POLICIES + 1,
        ):
            if rank <= len(top5):
                item = top5[rank - 1]

                result_row[
                    f"top{rank}_policy_id"
                ] = item["policy_id"]

                result_row[
                    f"top{rank}_title"
                ] = item["title"]

                result_row[
                    f"top{rank}_distance"
                ] = item["distance"]

                result_row[
                    f"top{rank}_region_scope"
                ] = item["region_scope"]

            else:
                result_row[
                    f"top{rank}_policy_id"
                ] = None

                result_row[
                    f"top{rank}_title"
                ] = None

                result_row[
                    f"top{rank}_distance"
                ] = None

                result_row[
                    f"top{rank}_region_scope"
                ] = None

    except Exception as error:
        print(
            f"처리 실패: {row['id']} "
            f"{row['region']} / {error}"
        )

        result_row = {
            "id": row["id"],
            "question": row["question"],
            "region": row["region"],
            "input_region_level1": None,
            "input_region_level2": None,
            "age": row["age"],
            "question_type": (
                row["question_type"]
            ),
            "expected_policy_id": (
                row["expected_policy_id"]
            ),
            "expected_policy_name": (
                row["answer"]
            ),
            "expected_rank": None,
            "hit_at_1": 0,
            "hit_at_3": 0,
            "hit_at_5": 0,
            "reciprocal_rank": 0.0,
            "region_error": str(error),
        }

    region_result_rows.append(
        result_row
    )

    if (
        (position + 1) % 20 == 0
        or position + 1 == len(save_df)
    ):
        print(
            "지역 필터 검색 완료:",
            f"{position + 1}/{len(save_df)}",
        )


region_filtered_results_df = pd.DataFrame(
    region_result_rows
)

print(
    "지역 필터 검색 완료 문항:",
    len(region_filtered_results_df),
)

print(
    "지역 처리 오류:",
    region_filtered_results_df[
        "region_error"
    ].notna().sum(),
)

display(
    region_filtered_results_df.head()
)

지역 필터 검색 완료: 20/280
지역 필터 검색 완료: 40/280
지역 필터 검색 완료: 60/280
지역 필터 검색 완료: 80/280
지역 필터 검색 완료: 100/280
지역 필터 검색 완료: 120/280
지역 필터 검색 완료: 140/280
지역 필터 검색 완료: 160/280
지역 필터 검색 완료: 180/280
지역 필터 검색 완료: 200/280
지역 필터 검색 완료: 220/280
지역 필터 검색 완료: 240/280
지역 필터 검색 완료: 260/280
지역 필터 검색 완료: 280/280
지역 필터 검색 완료 문항: 280
지역 처리 오류: 0


,id,question,region,input_region_level1,input_region_level2,age,question_type,expected_policy_id,expected_policy_name,expected_rank,...,top3_distance,top3_region_scope,top4_policy_id,top4_title,top4_distance,top4_region_scope,top5_policy_id,top5_title,top5_distance,top5_region_scope
0,Q081,대전에서 1인 영세 자영업자 고용보험료 지원을 받으려면 매출액과 가입 조건이 어떻게...,대전광역시,대전광역시,NaN,NaN,직접형,local:WLF00000877,1인 영세 자영업자 고용보험료 지원,1,...,0.440227,province,local:WLF00005253,대전 청년 월세지원,0.442353,province,national:WLF00000095,사회보험사각지대해소 사업(두루누리 사회보험료 지원사업),0.449528,national
1,Q082,대전에 사업장이 있고 연매출이 2억 원인 고용보험 가입 1인 자영업자입니다. 보험료...,대전광역시,대전광역시,NaN,NaN,간접형,local:WLF00000877,1인 영세 자영업자 고용보험료 지원,1,...,0.374317,national,national:WLF00005446,어업인 안전보험,0.375528,national,national:WLF00000095,사회보험사각지대해소 사업(두루누리 사회보험료 지원사업),0.386995,national
2,Q083,대전 구직청년 면접용 정장대여의 나이와 취업 상태 요건은 무엇이고 어디에서 신청하나요?,대전광역시,대전광역시,NaN,NaN,직접형,local:WLF00004054,구직청년 면접용 정장대여,1,...,0.369006,province,local:WLF00004332,청년 주택임차보증금 이자지원 사업,0.394472,province,local:WLF00005253,대전 청년 월세지원,0.413985,province
3,Q084,대전에 사는 25세 미취업 구직자입니다. 면접 정장이 필요한데 대전청년포털에서 인터...,대전광역시,대전광역시,NaN,25.0,간접형,local:WLF00004054,구직청년 면접용 정장대여,1,...,0.347372,province,local:WLF00005253,대전 청년 월세지원,0.353498,province,local:WLF00004332,청년 주택임차보증금 이자지원 사업,0.358003,province
4,Q085,대전 청년 월세지원의 연령·무주택·보증금·월세·소득 요건과 온라인 신청처를 알려주세요.,대전광역시,대전광역시,NaN,NaN,직접형,local:WLF00005253,대전 청년 월세지원,1,...,0.274315,province,national:WLF00001063,주거안정 월세대출,0.306892,national,local:WLF00000884,미래두배 청년통장(구 청년희망통장),0.327527,province


In [26]:
def metric_summary(df, search_method):
    return {
        "검색 방식": search_method,
        "문항 수": len(df),
        "Hit@1": df["hit_at_1"].mean(),
        "Hit@3": df["hit_at_3"].mean(),
        "Hit@5": df["hit_at_5"].mean(),
        "MRR": df[
            "reciprocal_rank"
        ].mean(),
    }


summary_rows = []

# 전체 비교
summary_rows.append(
    metric_summary(
        vector_results_df,
        "필터 없음 - 전체",
    )
)

summary_rows.append(
    metric_summary(
        region_filtered_results_df,
        "지역 필터 - 전체",
    )
)

# 직접형·간접형 비교
for question_type in [
    "직접형",
    "간접형",
]:
    before_subset = vector_results_df[
        vector_results_df[
            "question_type"
        ] == question_type
    ]

    after_subset = (
        region_filtered_results_df[
            region_filtered_results_df[
                "question_type"
            ] == question_type
        ]
    )

    summary_rows.append(
        metric_summary(
            before_subset,
            f"필터 없음 - {question_type}",
        )
    )

    summary_rows.append(
        metric_summary(
            after_subset,
            f"지역 필터 - {question_type}",
        )
    )


comparison_summary_df = pd.DataFrame(
    summary_rows
)

print("[지역 필터 전후 검색 성능]")
display(
    comparison_summary_df.style.format(
        {
            "Hit@1": "{:.4f}",
            "Hit@3": "{:.4f}",
            "Hit@5": "{:.4f}",
            "MRR": "{:.4f}",
        }
    )
)


# 문항별 순위 변화 비교
rank_comparison_df = (
    vector_results_df[
        [
            "id",
            "question",
            "question_type",
            "expected_policy_id",
            "expected_policy_name",
            "expected_rank",
            "hit_at_1",
        ]
    ]
    .rename(
        columns={
            "expected_rank": (
                "expected_rank_before"
            ),
            "hit_at_1": (
                "hit_at_1_before"
            ),
        }
    )
    .merge(
        region_filtered_results_df[
            [
                "id",
                "expected_rank",
                "hit_at_1",
                "input_region_level1",
                "input_region_level2",
                "top1_policy_id",
                "top1_title",
                "top1_region_scope",
                "region_error",
            ]
        ].rename(
            columns={
                "expected_rank": (
                    "expected_rank_after"
                ),
                "hit_at_1": (
                    "hit_at_1_after"
                ),
            }
        ),
        on="id",
        how="left",
    )
)


# 검색되지 않은 정책은 비교용으로 999위 처리
before_rank_for_compare = (
    rank_comparison_df[
        "expected_rank_before"
    ].fillna(999)
)

after_rank_for_compare = (
    rank_comparison_df[
        "expected_rank_after"
    ].fillna(999)
)

rank_comparison_df["rank_change"] = (
    before_rank_for_compare
    - after_rank_for_compare
)

rank_comparison_df["result_change"] = (
    "동일"
)

rank_comparison_df.loc[
    rank_comparison_df[
        "rank_change"
    ] > 0,
    "result_change",
] = "개선"

rank_comparison_df.loc[
    rank_comparison_df[
        "rank_change"
    ] < 0,
    "result_change",
] = "하락"


print(
    "\n순위 개선 문항:",
    (
        rank_comparison_df[
            "result_change"
        ] == "개선"
    ).sum(),
)

print(
    "순위 동일 문항:",
    (
        rank_comparison_df[
            "result_change"
        ] == "동일"
    ).sum(),
)

print(
    "순위 하락 문항:",
    (
        rank_comparison_df[
            "result_change"
        ] == "하락"
    ).sum(),
)


print("\n[지역 필터 적용 후 개선된 문항]")
display(
    rank_comparison_df[
        rank_comparison_df[
            "result_change"
        ] == "개선"
    ].sort_values(
        "rank_change",
        ascending=False,
    ).head(20)
)


print("\n[지역 필터 적용 후 하락한 문항]")
display(
    rank_comparison_df[
        rank_comparison_df[
            "result_change"
        ] == "하락"
    ].sort_values(
        "rank_change",
        ascending=True,
    ).head(20)
)


# 결과 저장
REGION_FILTER_RESULT_CSV = (
    OUTPUT_DIR
    / "rag_vector_region_filtered_results.csv"
)

REGION_COMPARISON_CSV = (
    OUTPUT_DIR
    / "rag_vector_region_comparison.csv"
)

REGION_SUMMARY_CSV = (
    OUTPUT_DIR
    / "rag_vector_region_summary.csv"
)

region_filtered_results_df.to_csv(
    REGION_FILTER_RESULT_CSV,
    index=False,
    encoding="utf-8-sig",
)

rank_comparison_df.to_csv(
    REGION_COMPARISON_CSV,
    index=False,
    encoding="utf-8-sig",
)

comparison_summary_df.to_csv(
    REGION_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)

print(
    "\n지역 필터 결과:",
    REGION_FILTER_RESULT_CSV,
)

print(
    "문항별 비교 결과:",
    REGION_COMPARISON_CSV,
)

print(
    "성능 요약 결과:",
    REGION_SUMMARY_CSV,
)

[지역 필터 전후 검색 성능]


,검색 방식,문항 수,Hit@1,Hit@3,Hit@5,MRR
0,필터 없음 - 전체,280,0.7857,0.9000,0.9357,0.8492
1,지역 필터 - 전체,280,0.8857,0.9607,0.9750,0.9253
2,필터 없음 - 직접형,140,0.9143,0.9714,0.9857,0.9441
3,지역 필터 - 직접형,140,0.9714,1.0000,1.0000,0.9857
4,필터 없음 - 간접형,140,0.6571,0.8286,0.8857,0.7542
5,지역 필터 - 간접형,140,0.8000,0.9214,0.9500,0.8648



순위 개선 문항: 46
순위 동일 문항: 234
순위 하락 문항: 0

[지역 필터 적용 후 개선된 문항]


,id,question,question_type,expected_policy_id,expected_policy_name,expected_rank_before,hit_at_1_before,expected_rank_after,hit_at_1_after,input_region_level1,input_region_level2,top1_policy_id,top1_title,top1_region_scope,region_error,rank_change,result_change
201,Q162,공주시 대출관련 지원정책 찾아줘,간접형,local:WLF00006616,공주시 다자녀가구 주택구입 및 전세자금 대출이자 지원사업,NaN,0,1,1,충청남도,공주시,local:WLF00006616,공주시 다자녀가구 주택구입 및 전세자금 대출이자 지원사업,district,None,998.0,개선
135,Q136,판돈으로 집갈 돈 다 날렸다. 좀 도와줘,간접형,local:WLF00003343,행려자 귀향여비 지원,NaN,0,10,0,경상남도,의령군,national:WLF00003271,내집마련 디딤돌 대출,national,None,989.0,개선
139,Q140,"최근 다쳐서 장애등급을 판정 받았는데, 제게 해당하는 정책이 있을까요?",간접형,local:WLF00000244,장애인의 날 행사 지원,NaN,0,35,0,경상남도,창녕군,local:WLF00003966,도비 장애수당 지원,province,None,964.0,개선
195,Q036,경기도 거주 갈 곳이 없어서 잠까 머물 곳이 필요한 상황인데 지원받을 수 있는 곳이...,간접형,local:WLF00003696,노숙인 일시보호소 운영,NaN,0,53,0,경기도,NaN,local:WLF00000715,노숙인 임시주거 지원,province,None,946.0,개선
219,Q180,나 강북구 사는 김첨지인데 배고파 죽겠어,간접형,local:WLF00005663,고독사 예방 밑반찬 지원사업 기운찬,64.0,0,4,0,서울특별시,강북구,national:WLF00006248,직장인 든든한 점심밥,national,None,60.0,개선
277,Q078,전세사기를 당해서 많이 힘들어… 주택이나 임대주택관련 복지정책이 없을까?,간접형,national:WLF00000062,기존주택등 매입임대주택 지원사업,31.0,0,3,0,NaN,NaN,national:WLF00003222,버팀목전세자금대출,national,None,28.0,개선
269,Q070,집이 없는 상황인데 국가사업 중에 무주택 관련 지원 있나?,간접형,national:WLF00000057,공공분양,29.0,0,4,0,NaN,NaN,national:WLF00003271,내집마련 디딤돌 대출,national,None,25.0,개선
124,Q125,"최근 김해에서 신생아를 출생한 산모인데, 해당하는 지원 정책이 뭐가 있을까요?",직접형,local:WLF00006457,유축기 대여 지원 사업,27.0,0,2,0,경상남도,김해시,local:WLF00006005,김해시 장애인가정 출산지원금 지원,district,None,25.0,개선
207,Q168,나 나 공주시에서 노숙자 됐는데 집가고싶어,간접형,local:WLF00005520,행려자 귀향여비 지원사업,29.0,0,6,0,충청남도,공주시,local:WLF00002298,노숙인 쉼터 운영지원,province,None,23.0,개선
9,Q090,대전 유성구에서 귀향할 교통비가 없는 노숙인입니다. 버스나 기차 승차권을 지원받으려...,간접형,local:WLF00001253,행려자지원,16.0,0,1,1,대전광역시,유성구,local:WLF00001253,행려자지원,district,None,15.0,개선



[지역 필터 적용 후 하락한 문항]


,id,question,question_type,expected_policy_id,expected_policy_name,expected_rank_before,hit_at_1_before,expected_rank_after,hit_at_1_after,input_region_level1,input_region_level2,top1_policy_id,top1_title,top1_region_scope,region_error,rank_change,result_change



지역 필터 결과: C:\ai_project01\WelfareOne\data\evaluation\rag_vector_region_filtered_results.csv
문항별 비교 결과: C:\ai_project01\WelfareOne\data\evaluation\rag_vector_region_comparison.csv
성능 요약 결과: C:\ai_project01\WelfareOne\data\evaluation\rag_vector_region_summary.csv
